# SCA-1 — SigLIP2 standalone complementarity

Bounded diagnostic over `DEV_CROSS_60`: A0 is frozen G1 OpenAI CLIP; S1 changes only grounding to pinned SigLIP2 over byte-identical OPUS-English text and the exact frozen 177,321 BTC catalog rows. U1 is post-GT oracle analysis only. No fusion or production-policy change.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZIP_STORED, ZipFile, ZipInfo

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
DATA_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
TEAM_EVAL_INPUT = Path(os.environ.get("AIC_TEAM_EVAL_DEV_ROOT", "/kaggle/input/datasets/irthn1311/aic2026_team_eval_dev_v1"))
STAGE1_INPUT = Path(os.environ.get("AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle"))
STAGE1B_INPUT = Path(os.environ.get("AIC_STAGE1B_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports"))
STAGE1E_INPUT = Path(os.environ.get("AIC_STAGE1E_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze"))
CLIP_INPUT = Path(os.environ.get("AIC_CLIP_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32"))
OPUS_INPUT = Path(os.environ.get("AIC_OPUS_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en"))
FREEZE_INPUT = Path(os.environ.get("AIC_SCA1_FREEZE_ROOT", "/kaggle/input/datasets/irthn1311/sca1-preparation-freeze-2026-08-17"))
SIGLIP_INPUT = Path(os.environ.get("AIC_SIGLIP2_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-siglip2-base-patch16-224"))
INDEX_INPUT = Path(os.environ.get("AIC_SCA1_INDEX_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-sca1-siglip2-index-v01"))
SIGLIP_DEVICE = os.environ.get("AIC_SIGLIP2_DEVICE", "auto")
SIGLIP_BATCH_SIZE = int(os.environ.get("AIC_SIGLIP2_BATCH_SIZE", "64"))
OUTPUT_ROOT = Path("/kaggle/working/artifacts/sca1_siglip2_complementarity_v01")
WORK_ROOT = Path("/kaggle/working/triage_eg_sca1_work")
INDEX_OUTPUT_ROOT = Path("/kaggle/working/sca1_siglip2_index_v01")
INDEX_ZIP_PATH = Path("/kaggle/working/triage_eg_sca1_siglip2_index_v01.zip")
ZIP_PATH = Path("/kaggle/working/triage_eg_sca1_siglip2_complementarity_v01_bundle.zip")
for target in (OUTPUT_ROOT, WORK_ROOT, INDEX_OUTPUT_ROOT):
    if target.exists():
        if Path("/kaggle/working") not in target.parents:
            raise RuntimeError(f"Refusing cleanup outside /kaggle/working: {target}")
        shutil.rmtree(target)
for target in (INDEX_ZIP_PATH, ZIP_PATH):
    target.unlink(missing_ok=True)
print({
    "required_inputs": {
        "raw_dataset": str(DATA_INPUT),
        "team_eval_dev_bundle": str(TEAM_EVAL_INPUT),
        "stage1_exact_index": str(STAGE1_INPUT),
        "stage1b_verified_contract": str(STAGE1B_INPUT),
        "stage1e_language_contract": str(STAGE1E_INPUT),
        "openai_clip_offline_asset": str(CLIP_INPUT),
        "opus_mt_vi_en_offline_asset": str(OPUS_INPUT),
        "sca1_preparation_freeze": str(FREEZE_INPUT),
        "siglip2_offline_asset": str(SIGLIP_INPUT),
        "optional_prebuilt_siglip2_index": str(INDEX_INPUT),
    },
    "internet_required": "ONLY_FOR_GIT_CLONE_IF_REPO_NOT_PRESENT",
    "model_download_required": False,
    "gpu_policy": "SIGLIP2_IMAGE_TEXT_AUTO; OPENAI_CLIP_AND_TRANSLATOR_EXISTING_GPU_CONFIG",
    "output_zip": str(ZIP_PATH),
    "index_zip_if_built": str(INDEX_ZIP_PATH),
})


In [ ]:
SEARCH_ROOT = Path("/kaggle/input")
MAX_DEPTH, MAX_DIRECTORIES = 7, 10000
def bounded_dirs(root):
    queue, visited = [(Path(root), 0)], 0
    while queue:
        current, depth = queue.pop(0)
        if not current.is_dir():
            continue
        visited += 1
        if visited > MAX_DIRECTORIES:
            raise RuntimeError("Kaggle input discovery exceeded bound")
        yield current
        if depth < MAX_DEPTH:
            queue.extend(
                (child, depth + 1)
                for child in sorted(current.iterdir())
                if child.is_dir() and not child.is_symlink()
            )
def named_mount_roots(hint):
    hint = Path(hint)
    candidates = (
        hint,
        Path('/kaggle/input') / hint.name,
        Path('/kaggle/input/datasets/irthn1311') / hint.name,
    )
    return list(dict.fromkeys(path for path in candidates if path.exists()))
def resolve_file(hint, filename, optional=False):
    hint = Path(hint)
    if hint.is_file() and hint.name == filename:
        return hint.resolve()
    roots, matches = named_mount_roots(hint), []
    if not roots:
        if optional:
            return None
        raise RuntimeError(
            f'Required Kaggle dataset is not attached for {filename}. '
            f'Expected mount slug {hint.name!r}; set the corresponding AIC_*_ROOT override if renamed.'
        )
    for root in roots:
        matches.extend(
            directory / filename
            for directory in bounded_dirs(root)
            if (directory / filename).is_file()
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one {filename}; found {matches}")
    return matches[0]
def resolve_root(hint, marker, optional=False):
    hint, marker = Path(hint), Path(marker)
    roots, matches = named_mount_roots(hint), []
    if not roots:
        if optional:
            return None
        raise RuntimeError(
            f'Required Kaggle dataset is not attached for marker {marker}. '
            f'Expected mount slug {hint.name!r}; set the corresponding AIC_*_ROOT override if renamed.'
        )
    for root in roots:
        matches.extend(
            directory
            for directory in bounded_dirs(root)
            if (directory / marker).is_file()
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one root with {marker}; found {matches}")
    return matches[0]
def resolve_dataset(hint):
    hint, marker = Path(hint), Path("map-keyframes-aic25-b1/map-keyframes")
    roots, matches = ([hint] if hint.exists() else [SEARCH_ROOT]), []
    for root in roots:
        matches.extend(
            directory
            for directory in bounded_dirs(root)
            if (directory / marker).is_dir() and any(directory.glob("Videos_*/video"))
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one raw dataset root; found {matches}")
    return matches[0]

DATASET_ROOT = resolve_dataset(DATA_INPUT)
TEAM_EVAL_ZIP_MOUNT = resolve_file(TEAM_EVAL_INPUT, "aic2026_team_eval_dev_v1.zip", optional=True)
TEAM_EVAL_ROOT_MOUNT = (
    None
    if TEAM_EVAL_ZIP_MOUNT
    else resolve_root(TEAM_EVAL_INPUT, "benchmarks/dev_cross_60/queries.jsonl")
)
FREEZE_ZIP = resolve_file(FREEZE_INPUT, "SCA1_PREPARATION_FREEZE_2026-08-17.zip")
PREBUILT_INDEX_ROOT = resolve_root(
    INDEX_INPUT, "index/siglip2_vectors.f16.npy", optional=True
)
print({
    "resolved_raw": str(DATASET_ROOT),
    "resolved_team_eval_zip": str(TEAM_EVAL_ZIP_MOUNT) if TEAM_EVAL_ZIP_MOUNT else None,
    "resolved_team_eval_root": str(TEAM_EVAL_ROOT_MOUNT) if TEAM_EVAL_ROOT_MOUNT else None,
    "resolved_sca1_freeze": str(FREEZE_ZIP),
    "resolved_optional_index": str(PREBUILT_INDEX_ROOT) if PREBUILT_INDEX_ROOT else None,
})


In [ ]:
if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f"Incomplete repository directory: {REPO_DIR}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
        env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
    )
module_file = REPO_DIR / "src/triage_eg/diagnostics/sca1_siglip2_complementarity/runner.py"
if not module_file.is_file():
    raise RuntimeError("TRIAGEEG ref does not contain SCA-1; publish reviewed source before Kaggle execution")
sys.path.insert(0, str(REPO_DIR / "src"))
HEAD = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
BRANCH = subprocess.run(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
GIT_STATUS = subprocess.run(
    ["git", "status", "--short"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
anchor = "cd43d1d2b33d4a5fa808f8f3aad23b199325e119"
ancestor = subprocess.run(
    ["git", "merge-base", "--is-ancestor", anchor, HEAD], cwd=REPO_DIR, check=False
).returncode == 0
if BRANCH != REPO_REF or not ancestor:
    raise RuntimeError(f"SCA1 repository lineage gate failed: branch={BRANCH}, anchor_ancestor={ancestor}")
print({"branch": BRANCH, "HEAD": HEAD, "git_status": GIT_STATUS or "CLEAN", "tca1_anchor_is_ancestor": ancestor})


In [ ]:
import yaml
from triage_eg.diagnostics.sca1_siglip2_complementarity import (
    SCA1Settings,
    load_preparation_freeze,
    validate_offline_asset,
)
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1d.inputs import resolve_input_root

MATERIALIZED = {
    name: WORK_ROOT / name
    for name in ("stage1", "stage1b", "stage1e", "clip", "opus", "siglip2")
}
def optional_search_root(hint):
    return None if Path(hint).exists() else SEARCH_ROOT
STAGE1_ROOT = resolve_stage1_root(
    STAGE1_INPUT,
    search_root=optional_search_root(STAGE1_INPUT),
    materialize_root=MATERIALIZED["stage1"],
)
STAGE1B_ROOT, _ = resolve_input_root(
    STAGE1B_INPUT,
    required=("stage1b_summary.json", "encoder/selected_encoder_contract.json", "encoder/runtime_adapter_manifest.json"),
    materialize_root=MATERIALIZED["stage1b"],
    search_root=optional_search_root(STAGE1B_INPUT),
    archive_keyword="stage1b",
)
STAGE1E_ROOT, _ = resolve_input_root(
    STAGE1E_INPUT,
    required=("stage1e_summary.json", "language_path_contract.json"),
    materialize_root=MATERIALIZED["stage1e"],
    search_root=optional_search_root(STAGE1E_INPUT),
    archive_keyword="stage1e",
)
CLIP_ROOT, _ = resolve_input_root(
    CLIP_INPUT,
    required=("checkpoint/ViT-B-32.pt", "manifests/asset_manifest.json"),
    materialize_root=MATERIALIZED["clip"],
    search_root=optional_search_root(CLIP_INPUT),
    archive_keyword="clip",
)
OPUS_ROOT, _ = resolve_input_root(
    OPUS_INPUT,
    required=("model/config.json", "manifests/asset_manifest.json"),
    materialize_root=MATERIALIZED["opus"],
    search_root=optional_search_root(OPUS_INPUT),
    archive_keyword="opus",
)
SIGLIP_ROOT, _ = resolve_input_root(
    SIGLIP_INPUT,
    required=("model/model.safetensors", "manifests/asset_manifest.json"),
    materialize_root=MATERIALIZED["siglip2"],
    search_root=optional_search_root(SIGLIP_INPUT),
    archive_keyword="siglip2",
)
SETTINGS = SCA1Settings()
EXPERIMENT_CONFIG = yaml.safe_load(
    (REPO_DIR / "configs/experiments/triage_sca1_siglip2_complementarity_v01.yaml").read_text(encoding="utf-8")
)
if (
    EXPERIMENT_CONFIG.get("experiment") != "TRIAGE_SCA1_SIGLIP2_COMPLEMENTARITY"
    or EXPERIMENT_CONFIG.get("selected_variant") != SETTINGS.variant
    or EXPERIMENT_CONFIG.get("production_policy_changed") is not False
    or EXPERIMENT_CONFIG.get("fusion_implemented") is not False
):
    raise RuntimeError("SCA1 YAML/dataclass contract mismatch")
PREPARATION = load_preparation_freeze(FREEZE_ZIP)
ASSET_VALIDATION = validate_offline_asset(SIGLIP_ROOT)
print({
    "stage1": str(STAGE1_ROOT),
    "stage1b": str(STAGE1B_ROOT),
    "stage1e": str(STAGE1E_ROOT),
    "clip": str(CLIP_ROOT),
    "opus": str(OPUS_ROOT),
    "siglip2": str(SIGLIP_ROOT),
    "freeze_validation": PREPARATION.validation,
    "asset_validation": ASSET_VALIDATION,
})


In [ ]:
TEST_COMMAND = [
    sys.executable,
    "-m",
    "pytest",
    "tests/unit/sca1_siglip2_complementarity",
    "tests/unit/d1_grounding_attribution",
    "tests/unit/e2eg1",
    "tests/unit/stage2_runtime",
    "-q",
]
test_process = subprocess.run(
    TEST_COMMAND, cwd=REPO_DIR, capture_output=True, text=True, check=False
)
test_text = (test_process.stdout + "\n" + test_process.stderr).strip()
match = re.search(r"(\d+) passed", test_text)
TEST_SUMMARY = {
    "command": " ".join(TEST_COMMAND),
    "returncode": test_process.returncode,
    "passed": int(match.group(1)) if match else None,
    "status": "PASS" if test_process.returncode == 0 else "FAIL",
    "output_tail": test_text.splitlines()[-30:],
}
print(TEST_SUMMARY)
if test_process.returncode != 0:
    raise RuntimeError("SCA1 required tests failed before experiment")


In [ ]:
from aic2026_eval.io import write_jsonl
from triage_eg.diagnostics.sca1_siglip2_complementarity import (
    Siglip2OfflineEncoder,
    build_siglip2_index,
    local_only_load_smoke,
    validate_siglip2_index,
)

QUERY_ONLY_ROOT = WORK_ROOT / "inference_only/dev_cross_60"
QUERY_ONLY_ROOT.mkdir(parents=True, exist_ok=True)
if TEAM_EVAL_ZIP_MOUNT:
    with ZipFile(TEAM_EVAL_ZIP_MOUNT) as archive:
        query_rows = [
            json.loads(line)
            for line in archive.read("benchmarks/dev_cross_60/queries.jsonl").decode("utf-8").splitlines()
            if line
        ]
else:
    query_rows = [
        json.loads(line)
        for line in (TEAM_EVAL_ROOT_MOUNT / "benchmarks/dev_cross_60/queries.jsonl").read_text(encoding="utf-8").splitlines()
        if line
    ]
write_jsonl(QUERY_ONLY_ROOT / "queries.jsonl", query_rows)
assert {path.name for path in QUERY_ONLY_ROOT.iterdir()} == {"queries.jsonl"}

LOCAL_ONLY_SMOKE = local_only_load_smoke(SIGLIP_ROOT)
SIGLIP_ENCODER = Siglip2OfflineEncoder(
    SIGLIP_ROOT, device=SIGLIP_DEVICE, batch_size=SIGLIP_BATCH_SIZE
).load()
INDEX_BUILT_THIS_RUN = PREBUILT_INDEX_ROOT is None
if INDEX_BUILT_THIS_RUN:
    INDEX_VALIDATION = build_siglip2_index(
        encoder=SIGLIP_ENCODER,
        dataset_root=DATASET_ROOT,
        stage1_root=STAGE1_ROOT,
        output_root=INDEX_OUTPUT_ROOT,
        batch_size=SIGLIP_BATCH_SIZE,
        git_commit=HEAD,
    )
    INDEX_ROOT = INDEX_OUTPUT_ROOT
    with ZipFile(INDEX_ZIP_PATH, "w", ZIP_STORED, allowZip64=True) as archive:
        for path in sorted(item for item in INDEX_ROOT.rglob("*") if item.is_file()):
            archive.write(path, path.relative_to(INDEX_ROOT).as_posix())
else:
    INDEX_ROOT = PREBUILT_INDEX_ROOT
    INDEX_VALIDATION = validate_siglip2_index(INDEX_ROOT, stage1_root=STAGE1_ROOT)
print({
    "GT_AVAILABLE_TO_INDEX_BUILD": False,
    "local_only_siglip2_smoke": LOCAL_ONLY_SMOKE,
    "index_built_this_run": INDEX_BUILT_THIS_RUN,
    "index_root": str(INDEX_ROOT),
    "index_manifest": INDEX_VALIDATION["manifest"],
    "index_zip": str(INDEX_ZIP_PATH) if INDEX_BUILT_THIS_RUN else None,
})


In [ ]:
from triage_eg.diagnostics.sca1_siglip2_complementarity import (
    Siglip2ExactBackend,
    Siglip2GroundingPipeline,
)
from triage_eg.e2eg1 import SafeCoveragePipeline
from triage_eg.retrieval.stage1b.adapters.openai_clip_official import (
    materialize_kaggle_expanded_tokenizer,
    resolve_official_asset_paths,
)
from triage_eg.retrieval.stage2 import OperationalRetrievalRuntime, config_from_yaml

CLIP_ASSET_PATHS = resolve_official_asset_paths(CLIP_ROOT)
SHARED_CLIP_SOURCE_ROOT, SHARED_CLIP_SOURCE_MATERIALIZED = materialize_kaggle_expanded_tokenizer(
    CLIP_ASSET_PATHS.source_root, WORK_ROOT / "shared_openai_clip_source"
)
os.environ["AIC_OPENAI_CLIP_SOURCE_ROOT"] = str(SHARED_CLIP_SOURCE_ROOT)
def make_runtime(name):
    config = config_from_yaml(
        REPO_DIR / "configs/retrieval/stage2_operational_runtime_gpu.yaml",
        stage1_root=STAGE1_ROOT,
        stage1b_root=STAGE1B_ROOT,
        stage1e_root=STAGE1E_ROOT,
        clip_asset_root=CLIP_ROOT,
        translator_asset_root=OPUS_ROOT,
        output_root=WORK_ROOT / f"runtime_{name}",
        stage1d_config=REPO_DIR / "configs/retrieval/stage1d_translation_ablation.yaml",
        build_git_commit=HEAD,
    )
    return OperationalRetrievalRuntime(config).load()
A0_RUNTIME = make_runtime("a0")
S1_RUNTIME = make_runtime("s1")
SIGLIP_BACKEND = Siglip2ExactBackend(INDEX_ROOT, stage1_root=STAGE1_ROOT)
A0_PIPELINE = SafeCoveragePipeline(A0_RUNTIME, DATASET_ROOT)
S1_PIPELINE = Siglip2GroundingPipeline(
    S1_RUNTIME,
    DATASET_ROOT,
    grounding_encoder=SIGLIP_ENCODER,
    grounding_backend=SIGLIP_BACKEND,
)
print({
    "GT_AVAILABLE_TO_PREDICTION": False,
    "shared_clip_source_root": str(SHARED_CLIP_SOURCE_ROOT),
    "shared_clip_source_materialized": SHARED_CLIP_SOURCE_MATERIALIZED,
    "runtime_identity_distinct": A0_RUNTIME is not S1_RUNTIME,
    "pipeline_identity_distinct": A0_PIPELINE is not S1_PIPELINE,
    "production_runtime_mutated": False,
})


In [ ]:
from triage_eg.diagnostics.sca1_siglip2_complementarity import (
    run_pre_gt_arm,
    validate_pre_gt_integrity,
)

A0_RUN, A0_SNAPSHOT = run_pre_gt_arm(
    A0_PIPELINE, QUERY_ONLY_ROOT, OUTPUT_ROOT, WORK_ROOT / "prediction_temp", "A0"
)
S1_RUN, S1_SNAPSHOT = run_pre_gt_arm(
    S1_PIPELINE, QUERY_ONLY_ROOT, OUTPUT_ROOT, WORK_ROOT / "prediction_temp", "S1"
)
INTEGRITY, TEXT_ROWS = validate_pre_gt_integrity(
    a0_run=A0_RUN,
    s1_run=S1_RUN,
    a0_snapshot=A0_SNAPSHOT,
    s1_snapshot=S1_SNAPSHOT,
    a0_pipeline=A0_PIPELINE,
    s1_pipeline=S1_PIPELINE,
    preparation=PREPARATION,
    siglip2_asset_root=SIGLIP_ROOT,
    siglip2_index_root=INDEX_ROOT,
    stage1_root=STAGE1_ROOT,
    settings=SETTINGS,
)
write_jsonl(OUTPUT_ROOT / "text_identity.jsonl", TEXT_ROWS)
print({
    "prediction_hashes": {"A0": A0_RUN["sha256"], "S1": S1_RUN["sha256"]},
    "integrity": INTEGRITY,
    "GT_OPENED": False,
})


In [ ]:
from triage_eg.e2e1 import extract_development_bundle

TEAM_EVAL_REPACKED = WORK_ROOT / "aic2026_team_eval_dev_v1_repacked.zip"
if TEAM_EVAL_ZIP_MOUNT:
    TEAM_EVAL_ZIP = TEAM_EVAL_ZIP_MOUNT
else:
    members = [
        path.relative_to(TEAM_EVAL_ROOT_MOUNT).as_posix()
        for path in TEAM_EVAL_ROOT_MOUNT.rglob("*")
        if path.is_file() and "sealed" not in path.as_posix().casefold()
    ]
    with ZipFile(TEAM_EVAL_REPACKED, "w", ZIP_DEFLATED) as archive:
        for member in sorted(members):
            info = ZipInfo(member, date_time=(1980, 1, 1, 0, 0, 0))
            info.compress_type = ZIP_DEFLATED
            info.external_attr = 0o644 << 16
            archive.writestr(info, (TEAM_EVAL_ROOT_MOUNT / member).read_bytes())
    TEAM_EVAL_ZIP = TEAM_EVAL_REPACKED
TEAM_EVAL_ROOT = extract_development_bundle(TEAM_EVAL_ZIP, WORK_ROOT / "team_eval_extracted")
CROSS_ROOT = TEAM_EVAL_ROOT / "benchmarks/dev_cross_60"
print({"GT_OPENED_AFTER_BOTH_HASHES": True, "cross_root": str(CROSS_ROOT)})


In [ ]:
from triage_eg.diagnostics.sca1_siglip2_complementarity import (
    create_bundle,
    evaluate_post_gt,
    formal_report,
    write_manifests,
)

EVALUATION = evaluate_post_gt(
    a0_pipeline=A0_PIPELINE,
    s1_pipeline=S1_PIPELINE,
    a0_run=A0_RUN,
    s1_run=S1_RUN,
    a0_snapshot=A0_SNAPSHOT,
    s1_snapshot=S1_SNAPSHOT,
    integrity=INTEGRITY,
    benchmark_root=CROSS_ROOT,
    output_root=OUTPUT_ROOT,
    temporary_root=WORK_ROOT / "post_gt",
)
RESOLVED_INPUTS = {
    "raw_dataset": str(DATASET_ROOT),
    "team_eval_dev_bundle": str(TEAM_EVAL_ZIP),
    "stage1": str(STAGE1_ROOT),
    "stage1b": str(STAGE1B_ROOT),
    "stage1e": str(STAGE1E_ROOT),
    "clip": str(CLIP_ROOT),
    "opus": str(OPUS_ROOT),
    "sca1_freeze": str(FREEZE_ZIP),
    "siglip2_asset": str(SIGLIP_ROOT),
    "siglip2_index": str(INDEX_ROOT),
}
write_manifests(
    OUTPUT_ROOT,
    settings=SETTINGS,
    preparation=PREPARATION,
    integrity=INTEGRITY,
    a0_run=A0_RUN,
    s1_run=S1_RUN,
    asset_root=SIGLIP_ROOT,
    index_root=INDEX_ROOT,
    git_commit=HEAD,
    branch=BRANCH,
    test_summary=TEST_SUMMARY,
    experiment_config={
        "yaml": EXPERIMENT_CONFIG,
        "resolved_inputs": RESOLVED_INPUTS,
        "index_built_this_run": INDEX_BUILT_THIS_RUN,
        "index_zip": str(INDEX_ZIP_PATH) if INDEX_BUILT_THIS_RUN else None,
    },
)
BUNDLE = create_bundle(OUTPUT_ROOT, ZIP_PATH)
print(formal_report(head=HEAD, integrity=INTEGRITY, evaluation=EVALUATION, bundle=BUNDLE))
print("INPUTS_USED=", RESOLVED_INPUTS)
print("DOWNLOAD_ZIP=", ZIP_PATH)
print("INDEX_DOWNLOAD_ZIP=", INDEX_ZIP_PATH if INDEX_BUILT_THIS_RUN else "PREBUILT_INDEX_REUSED")
A0_PIPELINE.close()
S1_PIPELINE.close()
SIGLIP_ENCODER.close()
